# project_14_nanobody_viral — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — the conserved neutralizing epitope + VHH metrics

**Standard slot:** *define & explore.* **For Project 14 this means:** clean the viral antigen, pick a
**conserved neutralizing epitope** + a humanized VHH **framework**, fix the metrics, and run a tiny
mock VHH batch as your hello-world (D0).

> **Defensive framing.** Every design is steered to a conserved epitope to *block* the virus. Enhancing
> viral fitness/affinity/escape is out of scope (`README.md` → Responsible research, `MASTER_BLUEPRINT.md §7`).

Run `00_setup.ipynb` first in this session.

## The metrics, precisely (nanobody design)

| Metric | Means | Does **not** mean |
|--------|-------|-------------------|
| `pae_interaction` (AF2-Multimer) | interface confidence (antibody cutoff ≤ 12; lower better) | measured affinity |
| interface pLDDT | local confidence | stability / K_D |
| scRMSD | designed-vs-predicted VHH backbone self-consistency | binding |
| CDR geometry RMSD | loop/Ramachandran sanity vs IgFold | function |
| developability (TAP/CamSol/humanness) | aggregation/solubility/immunogenicity **proxies** | a verdict (use real tools) |
| **worst-case breadth pae** | does the VHH hold across ALL strains? | best-case is not breadth |


## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Epitope + framework choice (EXAMPLE — verify on the real structure)

Choose hotspots on the **conserved neutralizing face** (e.g. the HA stem). The residues below are
**EXAMPLE placeholders** — derive the real ones from 4FQI + a cross-strain conservation analysis.

In [ ]:
import antibody_tools as ab

ANTIGEN = "HA_STEM"
EPITOPE = ab.parse_epitope("A18,A41,A45,A49")   # EXAMPLE conserved HA-stem residues — VERIFY from 4FQI
FRAMEWORK = ab.DEFAULT_FRAMEWORK                  # humanized VHH placeholder — verify/replace
print("antigen:", ANTIGEN, "| EXAMPLE epitope (verify):", EPITOPE)
print("framework:", FRAMEWORK["name"])

## Hello-world: a tiny mock VHH batch

Develop the plumbing with the deterministic `mock` backend (no GPU). Switch `tool="rfantibody"` on an
A100 (see `MANUAL.md §2`). **Mock numbers/sequences are SYNTHETIC — never report them.**

In [ ]:
vhhs = ab.design_vhh_cdrs(ANTIGEN, EPITOPE, framework=FRAMEWORK, n=5, tool="mock")
ab.score_designs(vhhs, tool="mock")
d = vhhs[0]
print("example:", d.design_id, "| CDR3=", d.cdr3, "(", len(d.cdr3), "aa )")
print("  af2: pae_interaction=", d.pae_interaction, "scrmsd=", d.scrmsd, "cdr_geom=", d.cdr_geom)
print("  dev: tap=", d.tap_score, "camsol_like=", d.camsol_like, "humanness=", d.humanness, "(TEACHING HEURISTICS)")
print("  epitope overlap (neutralization proxy):", ab.epitope_overlap(d.contact_residues, EPITOPE))
print("[reminder] every number above is SYNTHETIC (mock).")

## Breadth concept: epitope conservation across strains `[core]`

A neutralizing VHH is broadly protective only if its epitope is **conserved** across strains. The
snippet below is **EXAMPLE_DATA** (toy aligned fragments) to demonstrate `epitope_conservation()`;
in your project you align a real strain panel and score the columns under your epitope.

In [ ]:
# EXAMPLE_DATA — toy aligned antigen fragments (NOT real sequences), positions 1..10.
EXAMPLE_STRAINS = {
    "H1": "GLFGAIAGFI",
    "H3": "GLFGAIAGFI",
    "H5": "GLFGAIAGFL",   # a change at position 10
}
stem_epitope = [2, 5, 8]   # toy conserved-stem positions
head_epitope = [10]        # toy variable-head position
print("conserved (stem) epitope conservation:", ab.epitope_conservation(stem_epitope, EXAMPLE_STRAINS))
print("variable (head) epitope conservation: ", ab.epitope_conservation(head_epitope, EXAMPLE_STRAINS))
print("[EXAMPLE_DATA] toy demo of the breadth rationale — replace with the real strain panel.")

## D0 checklist
- [ ] Conserved-epitope map + justification of the chosen neutralizing face; humanized framework chosen.
- [ ] One-paragraph definition of each metric **with** its 'does not mean' note.
- [ ] One reproduced mock mini-run (VHHs scored, developability + breadth rationale shown).
- [ ] `LOG.md` entry: tool version, GPU, seed.

**Next:** `02_generate.ipynb` — the RFantibody CDR design campaign.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — RFantibody CDR design at the conserved epitope

**Standard slot:** *design campaign.* **For Project 14 this means:** design VHH CDRs against the
conserved neutralizing epitope and assemble the pool (D2). Hit rates are LOW → generate **500+** at
real scale; survivors are **screening inputs**. The real campaign wants an **A100** — develop on `mock`.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Verify the upstreams still exist (pin commits — they change)

In [ ]:
import requests
UPSTREAMS = {
    "RFantibody":    "https://github.com/RosettaCommons/RFantibody",   # pin a commit; VERIFY
    "ImmuneBuilder": "https://github.com/oxpig/ImmuneBuilder",         # CDR geometry
    "ColabFold":     "https://github.com/sokrypton/ColabFold",         # AF2-Multimer
    "ProteinMPNN":   "https://github.com/dauparas/ProteinMPNN",
}
# BoltzGen [extension]: verify the current public release at course start and add it here.
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, timeout=20, allow_redirects=True)
        print(f"{name:14s} {r.status_code}  {url}")
    except Exception as e:
        print(f"{name:14s} ERR  {url}  ({e})")

## Run the CDR design campaign (mock; switch tool= on an A100)

RFantibody CDR design (aim 500+ at real scale) → AF2-Multimer + IgFold + developability. Here we use a
small mock count so the notebook runs anywhere. All numbers are **SYNTHETIC**.

In [ ]:
import antibody_tools as ab, pandas as pd

ANTIGEN = "HA_STEM"
EPITOPE = ab.parse_epitope("A18,A41,A45,A49")   # EXAMPLE — verify
pool = ab.design_vhh_cdrs(ANTIGEN, EPITOPE, n=200, tool="mock")   # -> 500+ real (rfantibody)
ab.score_designs(pool, tool="mock")
df = pd.DataFrame([d.as_row() for d in pool])
df.to_csv("results/designs.csv", index=False)
print("VHH pool:", len(pool), "| wrote results/designs.csv", df.shape, "(all SYNTHETIC / EXAMPLE_DATA)")
df[["design_id", "cdr3", "pae_interaction", "scrmsd", "cdr_geom", "tap_score", "humanness"]].head(3)

## D2 checklist
- [ ] VHH pool (500+ at real scale) in `results/designs.csv`, with CDRs + metrics + developability.
- [ ] Design log: tool versions, params, seeds, runtimes in `LOG.md`.
- [ ] Interim report (3–4 pages); survivors framed as display-screen inputs.

**Next:** `03_filter_and_rank.ipynb` — the shared antibody filter.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — the shared multi-layer antibody filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 14** we run the **antibody** cutoffs on the VHH pool and report honest survival (D3 pt 1).

Run `00`–`02` first so `results/designs.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline

In [ ]:
import filtering_pipeline as fp
import pandas as pd
print("antibody cutoffs:", fp.DEFAULT_CUTOFFS["antibody"])

## Build `fp.Design` objects from the campaign

Map each VHH onto the shared `Design` record (`design_type="antibody"`). CDR geometry + developability
ride along in `extra`. Layer 2 (orthogonal) needs a *second* predictor (IgFold/ESMFold) — we run
layers **(1, 3)** here and note L2 is added with a real second predictor.

In [ ]:
df = pd.read_csv("results/designs.csv")
designs = [fp.Design(design_id=str(r.design_id), sequence=str(r.sequence), design_type="antibody",
                     plddt=r.plddt, pae_interaction=r.pae_interaction, scrmsd=r.scrmsd,
                     extra={"cdr_geom": r.cdr_geom, "tap_score": r.tap_score,
                            "humanness": r.humanness, "camsol_like": r.camsol_like, "synthetic": True})
           for r in df.itertuples()]
print(len(designs), "Design objects built (design_type='antibody')")

## Run the pipeline + report

In [ ]:
ranked = fp.run_pipeline(designs, design_type="antibody", use_layers=(1, 3))
ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(ranked, top_n=10, save_prefix="results/proj14")
print("\nNOTE: numbers are SYNTHETIC (mock). Layer 2 (orthogonal IgFold/ESMFold) is added in a real run.")
top

## Survival-at-each-layer (honest accounting)
Report N pass / N generated at each layer. De novo antibody hit rates are LOW — survivors are screening inputs.

In [ ]:
if "layers_passed" in ranked:
    print(ranked["layers_passed"].value_counts().sort_index())

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (`design_type='antibody'`).
- [ ] Survival-at-each-layer reported (honest, low hit rate).
- [ ] Mapping assumptions written down.

**Next:** `04_validate.ipynb` — developability gate + cross-strain breadth.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — developability gate + cross-strain breadth

**Standard slot:** *validate (in silico).* **For Project 14 the core science is BREADTH + developability:**
does each top VHH hold across a strain panel, and is it developable? Report the **worst-case** strain (D3 pt 2).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Developability gate (teaching heuristics — swap in real tools)

Filter survivors on the developability **proxies** (lower `tap_score`, higher `humanness`). These are
CLEARLY-LABELED teaching heuristics, NOT validated TAP/CamSol/Hu-mAb — replace before any conclusion.

In [ ]:
import pandas as pd
ranked = pd.read_csv("results/ranked.csv")
designs = pd.read_csv("results/designs.csv")[["design_id", "tap_score", "humanness", "camsol_like", "cdr3"]]
df = ranked.merge(designs, on="design_id", how="left")
TAP_MAX, HUMAN_MIN = 6.0, 0.60   # EXAMPLE teaching thresholds — calibrate with real tools
dev_ok = df[(df["tap_score"] <= TAP_MAX) & (df["humanness"] >= HUMAN_MIN)]
print(f"developability gate (tap<={TAP_MAX}, humanness>={HUMAN_MIN}): {len(dev_ok)}/{len(df)} pass (SYNTHETIC)")
dev_ok[["design_id", "layers_passed", "pae_interaction", "tap_score", "humanness"]].head(8)

## 2 · Cross-strain breadth

Model each top candidate against the strain panel; record per-strain `pae_interaction`; rank by the
**worst-case** strain. The panel is **EXAMPLE_DATA** (mock); use your real verified strains.

In [ ]:
import antibody_tools as ab, numpy as np
top = (dev_ok if len(dev_ok) else df).head(10)
STRAINS = ["H1", "H3", "H5", "H7", "InfB"]   # EXAMPLE panel — verify/replace
rows = []
for r in top.itertuples():
    seq = designs.set_index('design_id').loc[r.design_id, 'cdr3'] if False else str(r.sequence)
    prof = ab.breadth_across_strains(str(r.sequence), STRAINS, tool="mock")
    prof["design_id"] = r.design_id
    prof["worst_case_pae"] = max(v for v in prof.values() if isinstance(v, (int, float)))
    rows.append(prof)
breadth = pd.DataFrame(rows).set_index("design_id").sort_values("worst_case_pae")
breadth.to_csv("results/breadth.csv")
print("breadth profile (SYNTHETIC) — broadest (lowest worst-case pae) first:")
breadth

In [ ]:
import matplotlib.pyplot as plt
panel = [c for c in breadth.columns if c != "worst_case_pae"]
fig, ax = plt.subplots(figsize=(7, 4))
for did, row in breadth.iterrows():
    ax.plot(panel, [row[c] for c in panel], marker="o", alpha=0.6, label=str(did)[:18])
ax.axhline(12, ls="--", c="k", lw=0.8, label="antibody cutoff 12")
ax.set_ylabel("pae_interaction (lower = better)"); ax.set_xlabel("strain")
ax.set_title("Cross-strain breadth (EXAMPLE_DATA / mock)")
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig("results/proj14_breadth.png", dpi=150); plt.show()

## D3 (part 2) checklist
- [ ] Developability gate applied (real tools swapped in for any reported conclusion).
- [ ] Cross-strain breadth table + figure; **worst-case** strain reported per candidate.
- [ ] RFantibody vs BoltzGen head-to-head (hit rate, CDR geometry, developability) `[extension]`.
- [ ] Failure-mode notes: narrow/escape-prone or liability-laden designs.

**Next:** `05_validation_plan.ipynb` — the yeast-display screen + neutralization plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Yeast-display screen + neutralization plan (with biosafety oversight)

**Standard slot:** *validation plan.* **For Project 14 this means:** because de novo antibody hit rates
are low, the deliverable is a **yeast-display screen** that turns the pool into real binders, plus a
neutralization/breadth plan — under institutional biosafety oversight (D4/D5).

> **Defensive / biosafe.** Neutralization is tested with a **pseudovirus surrogate (standard BSL-2)** under
> **IBC approval** — never authentic-virus gain-of-function. See `MANUAL.md §7` and `MASTER_BLUEPRINT.md §7`.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Assemble the plan

Generated from your top breadth-and-developability candidates, written to `results/validation_plan.md`.

In [ ]:
import pandas as pd, os
breadth = pd.read_csv("results/breadth.csv") if os.path.exists("results/breadth.csv") else None
top_ids = list(breadth['design_id'][:8]) if breadth is not None else ['<top candidates from nb04>']
plan = f'''# Project 14 — Yeast-display screen + neutralization plan (DRAFT)

## Pooled library for display (top candidates; SYNTHETIC ids in this dry run)
{top_ids}

## Yeast-display screen (the workhorse for low-hit-rate de novo antibodies)
1. Synthesize the designed VHH pool; clone into a yeast-surface-display vector.       [build]
2. FACS against labeled conserved antigen; enrich binders over rounds.                [select]
3. Deep-sequence winners; pick a diverse panel; express solubly (E. coli/yeast).      [recover]
4. SPR/BLI vs antigen (K_D, kinetics); IgFold-check CDR geometry of winners.          [characterize]

## Neutralization + breadth (defensive; biosafe)
5. Pseudovirus neutralization across the STRAIN PANEL (IC50 per strain).              [breadth; BSL-2, IBC-approved]
   - Report IC50 for EACH strain; rank by WORST-CASE strain, not best.

## Controls (mandatory)
- Positive: a known neutralizing nanobody.
- Negative: scrambled-CDR variant of each candidate.
- Negative: an irrelevant-antigen VHH (specificity).

## Biosafety / responsible research
- Pseudovirus assays + any viral material: institutional biosafety committee (IBC) approval at the
  appropriate containment level. Gene synthesis via an IGSC biosecurity-screening provider.
- Defensive/neutralizing purpose only; no enhancement of viral fitness/affinity/escape (MASTER_BLUEPRINT §7).

## Cost + timeline
- [fill in library synthesis, display reagents, FACS time, SPR, pseudovirus panel, and a Gantt].
'''
open('results/validation_plan.md', 'w').write(plan)
print('wrote results/validation_plan.md')
print(plan[:950])

## D4 / D5 checklist
- [ ] `results/validation_plan.md`: yeast-display screen + neutralization/breadth + controls + costs.
- [ ] Biosafety/IBC pathway named; gene-synthesis screening noted; defensive framing explicit.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a low-hit-rate-aware, breadth-and-developability-filtered, defensively-framed VHH campaign.